# SignBridge — MMPose Batch Pose Extraction

This notebook runs on Google Colab (GPU runtime required).

It will:
1. Install MMPose/RTMPose + dependencies
2. Download RTMPose whole-body checkpoint
3. Pull ASLLVD videos from Watchtower via rsync
4. Run batch pose extraction on all 547 videos
5. Push results back to Watchtower

**Runtime: ~3 hours on T4 GPU**

---

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Cell 2: Install MMPose + dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -U openmim
!mim install -q mmengine
!mim install -q mmcv
!mim install -q mmdet
!mim install -q mmpose
!pip install -q pandas opencv-python ffmpeg-python
!apt-get -qq install -y ffmpeg
print('✅ MMPose installed')

In [ ]:
# Cell 3: Download RTMPose checkpoints
import os
os.makedirs('/content/checkpoints', exist_ok=True)
%cd /content/checkpoints

# RTMPose whole-body (133 keypoints)
!wget -q https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-wholebody_pt-aic-coco_270e-256x192-ea89636c_20230115.pth -O rtmpose-wholebody.pth
!wget -q https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-wholebody_pt-aic-coco_270e-256x192-ea89636c_20230115.json -O rtmpose-wholebody.json

# RTMDet for person detection
!wget -q https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_m_8xb32-300e_coco/rtmdet_m_8xb32-300e_coco_20220719_112220-229f2187.pth -O rtmdet.pth
!wget -q https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_m_8xb32-300e_coco/rtmdet_m_8xb32-300e_coco_20220719_112220-229f2187.py -O rtmdet.py

print('✅ Checkpoints downloaded')
!ls -lh /content/checkpoints/

In [ ]:
# Cell 4: Upload videos from Watchtower
# Option A: Mount Google Drive and upload there first
# Option B: Use gdown if videos are in Google Drive
# Option C: Direct upload (too large for 101GB)
# Option D: rsync from Watchtower (needs Colab SSH setup)

# Simplest: Upload videos + gloss map to Google Drive, then mount here
from google.colab import drive
drive.mount('/content/drive')

# After uploading signbridge/data/ to Google Drive:
# - data/raw/videos/ → /content/drive/MyDrive/signbridge/videos/
# - data/processed/gloss_video_map.csv → /content/drive/MyDrive/signbridge/gloss_video_map.csv

import os
VIDEO_DIR = '/content/drive/MyDrive/signbridge/videos'
GLOSS_MAP = '/content/drive/MyDrive/signbridge/gloss_video_map.csv'
OUTPUT_DIR = '/content/pose_library'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check files exist
if os.path.exists(VIDEO_DIR):
    sessions = os.listdir(VIDEO_DIR)
    print(f'✅ Video dir found: {len(sessions)} sessions')
else:
    print(f'❌ Video dir not found: {VIDEO_DIR}')
    print('Upload signbridge/data/raw/videos/ to your Google Drive first!')

In [ ]:
# Cell 5: Run batch pose extraction
# Paste the batch_pose_extraction.py code here or upload it

import cv2
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path

def extract_pose_video(video_path, output_path, start_frame, end_frame,
                       det_config, det_ckpt, pose_config, pose_ckpt):
    from mmpose.apis import init_model as init_pose_model
    from mmpose.apis import inference_topdown
    from mmdet.apis import init_detector, inference_detector
    from mmpose.apis import visualize_pose_prediction
    
    det_model = init_detector(det_config, det_ckpt, device='cuda:0')
    pose_model = init_pose_model(pose_config, pose_ckpt, device='cuda:0')
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return False, 'cannot open', 0
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_count = 0
    target_frames = end_frame - start_frame + 1
    
    while cap.isOpened() and frame_count < target_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        det_result = inference_detector(det_model, frame)
        pred = det_result.pred_instances
        
        if len(pred.scores) == 0:
            out.write(np.zeros_like(frame))
            frame_count += 1
            continue
        
        best_idx = pred.scores.argmax().item()
        bbox = pred.bboxes[best_idx].cpu().numpy()
        
        pose_results = inference_topdown(pose_model, frame, [bbox])
        
        black_bg = np.zeros_like(frame)
        vis_frame = visualize_pose_prediction(
            pose_model, black_bg, pose_results, radius=4, thickness=2
        )
        out.write(vis_frame)
        frame_count += 1
    
    cap.release()
    out.release()
    return True, f'{frame_count} frames', frame_count

# Load gloss map
gloss_map = pd.read_csv(GLOSS_MAP)
print(f'📋 {len(gloss_map)} glosses to process')

# Process
DET_CFG = '/content/checkpoints/rtmdet.py'
DET_CKPT = '/content/checkpoints/rtmdet.pth'
POSE_CFG = '/content/checkpoints/rtmpose-wholebody.json'
POSE_CKPT = '/content/checkpoints/rtmpose-wholebody.pth'

success = 0
fail = 0
failed_list = []
start_time = time.time()

for i, row in gloss_map.iterrows():
    gloss = row['gloss']
    session = str(row['session'])
    scene = int(row['scene'])
    start = int(row['start_frame'])
    end = int(row['end_frame'])
    
    video_path = f'{VIDEO_DIR}/{session}/scene{scene}-camera1.mov'
    output_path = f'{OUTPUT_DIR}/{gloss}.mp4'
    
    if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        success += 1
        continue
    
    if not os.path.exists(video_path):
        fail += 1
        failed_list.append(f'{gloss}: video not found')
        continue
    
    ok, msg, frames = extract_pose_video(
        video_path, output_path, start, end,
        DET_CFG, DET_CKPT, POSE_CFG, POSE_CKPT
    )
    
    if ok:
        success += 1
    else:
        fail += 1
        failed_list.append(f'{gloss}: {msg}')
    
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        eta = (len(gloss_map) - i - 1) / rate
        print(f'  [{i+1}/{len(gloss_map)}] ✅{success} ❌{fail} | {rate:.1f}/s | ETA: {eta:.0f}s')

elapsed = time.time() - start_time
print(f'\n📊 Done: {success} ok, {fail} failed, {elapsed/60:.1f} min')

if failed_list:
    with open(f'{OUTPUT_DIR}/failed.txt', 'w') as f:
        for line in failed_list:
            f.write(line + '\n')
    print(f'Failed: {OUTPUT_DIR}/failed.txt')

In [ ]:
# Cell 6: Build pose lookup table
lookup = {}
for _, row in gloss_map.iterrows():
    gloss = row['gloss']
    pose_path = f'{OUTPUT_DIR}/{gloss}.mp4'
    if os.path.exists(pose_path) and os.path.getsize(pose_path) > 1000:
        lookup[gloss] = {
            'pose_video': pose_path,
            'signer': row['signer'],
            'start_frame': int(row['start_frame']),
            'end_frame': int(row['end_frame']),
            'session': str(row['session']),
            'scene': int(row['scene'])
        }

with open(f'{OUTPUT_DIR}/pose_lookup.json', 'w') as f:
    json.dump(lookup, f, indent=2)

print(f'✅ Pose lookup: {len(lookup)} glosses')
print(f'   File: {OUTPUT_DIR}/pose_lookup.json')

In [ ]:
# Cell 7: Copy results to Google Drive for download
!mkdir -p /content/drive/MyDrive/signbridge/pose_library
!cp -r /content/pose_library/* /content/drive/MyDrive/signbridge/pose_library/
print('✅ Results copied to Google Drive')
print('   Download from: MyDrive/signbridge/pose_library/')
print('   Then rsync to Watchtower: ~/projects/signbridge/data/pose_library/')